Define the Schema

In [0]:
facilities_schema = """facility_id INT, facility_name STRING, member_cost DOUBLE, guest_cost DOUBLE, 
    initial_outlay DOUBLE, monthly_maintainance DOUBLE"""

members_schema = """member_id INT, last_name STRING, first_name STRING, address STRING, zip_code STRING, 
    telephone STRING, recommended_by STRING, joining_date DATE"""

bookings_schema = "booking_id INT, facility_id INT, member_id INT, start_time TIMESTAMP, slots INT"


####3. Load facilities table

In [0]:
facilities_df = (
    spark.read.format("csv")
    .schema(facilities_schema)
    .option("header", True)
    .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/facilities.csv")
)

facilities_df.write.mode("overwrite").option('OverwriteSchema', 'true').saveAsTable("dev.spark_db.facilities")


In [0]:
%sql

select * from facilities

####4. Load members table

In [0]:
members_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(members_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/members.csv")
)

members_df.write.mode("error").saveAsTable("dev.spark_db.members")

In [0]:
spark.sql("DROP TABLE IF EXISTS dev.spark_db.members")

In [0]:
bookings_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(bookings_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/bookings.csv")
)

bookings_df.write.mode("overwrite").saveAsTable("dev.spark_db.bookings")

Q1. Prepare a facility bookings reporting dataset as the following.
```
member_id | first_name | last_name | facility_id | slots | start_time
---------------------------------------------------------------------------
```
The report must meet the following criteria.
1. Facility bookings made by a person whose last name is Smith
2. He has booked more than 5 slots in a single booking
3. Report should be sorted by first name of the member in ascending order and number of slots in descending order


In [0]:
%sql
select m.member_id, first_name, last_name, facility_id, slots, start_time
from dev.spark_db.bookings as b inner join dev.spark_db.members as m on m.member_id = b.member_id
where m.last_name = "Smith" and b.slots > 5
order by m.first_name asc, b.slots desc

#### with dataframe API

In [0]:
from pyspark.sql.functions import col, expr

bookings_df = spark.table("dev.spark_db.bookings").alias("b")
members_df = spark.table("dev.spark_db.members").alias("m")
facilities_df = spark.table("dev.spark_db.facilities").alias("f")

report_df = (
    bookings_df
        .join(members_df, expr("b.member_id == m.member_id"), "inner")
        .join(facilities_df, col("b.facility_id") == col("f.facility_id"), "inner")
        .filter("m.last_name == 'Smith' and b.slots > 5")
        .selectExpr("m.member_id", "m.first_name", "m.last_name", "f.facility_name", "b.slots",
            "b.slots * f.member_cost as booking_amount", "b.start_time")
        .orderBy(col("m.first_name").asc(),
                 col("booking_amount").desc())
)

report_df.display()


In [0]:
result_df.display()